# Telco Customer Churn Prediction & Analysis

## Project Overview
The objective of this project is to predict customer churn for a telecommunications company and identify the key drivers behind customer attrition.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv("Telco-Customer-Churn.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

### 1. Data Cleaning: Handling TotalCharges
The `TotalCharges` column is stored as an `object` type because of whitespace entries for new customers. We convert it to a numeric type using `errors='coerce'`, which turns invalid strings into `NaN`.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors = "coerce")

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

Now we got 11 nulls in TotalCharges

In [ ]:
df[df["TotalCharges"].isnull()]

All 11 missing values correspond to customers with `tenure == 0` (newly joined customers who haven't received their first bill yet). Therefore, imputing these missing values with `0` is domain-appropriate.

In [ ]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [ ]:
df.nunique()

In [ ]:
df.duplicated().sum()

### 2. Feature Selection: Removing Unique Identifiers
`customerID` is an arbitrary identifier and does not contain predictive signal. It is dropped to prevent noise.

In [ ]:
df = df.drop('customerID', axis = 'columns')

In [ ]:
df.info()

Let's handle the binary columns. Turning them into ones and zeros. (partner, dependents,PhoneService, Paperlessbilling, churn)

In [ ]:
df.columns = df.columns.str.strip()

### 3. Simplifying Categorical Labels
Values such as `'No internet service'` and `'No phone service'` are redundant because internet and phone service availability are already tracked in separate columns. We map them to `'No'` to reduce categorical cardinality.

In [ ]:
df = df.replace({
    "No internet service": "No",
    "No phone service": "No"})

In [ ]:
df.head(20)

In [ ]:
df.nunique()

Changing every column that has the unique subset of "Yes" and "No" s into ones and zeros.

In [ ]:
binary_cols = [
    col for col in df.columns 
    if set(df[col].unique()).issubset({"Yes", "No"})
]
df[binary_cols] = df[binary_cols].replace({"No": 0, "Yes": 1})

In [ ]:
df.head(20)

Categorizing the columns, preparing them for the model

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in df.columns if c not in num_cols + ["Churn"]]

### 4. Exploratory Data Analysis (EDA)

In [ ]:
df["Churn"].value_counts(normalize=True)

Bivariate Analysis - how does a categorical variable affect the churn rate?

In [ ]:
for col in cat_cols:
    print(df.groupby(col)["Churn"].mean().sort_values(ascending=False) *100, "\n")

#### 4.1 Churn Rate by Contract, Internet Service, and Payment Method
Visualizing how contract length, internet service type, and billing methods impact customer retention.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
features = ['Contract', 'InternetService', 'PaymentMethod']

for ax, col in zip(axes, features):
    churn_rates = df.groupby(col)['Churn'].mean().reset_index()
    # hue=col ve legend=False eklendi:
    sns.barplot(data=churn_rates, x=col, y='Churn', ax=ax, hue=col, palette='Blues_r', legend=False)
    ax.set_title(f'Churn Rate by {col}', fontweight='bold')
    ax.set_ylabel('Churn Rate')
    ax.set_ylim(0, 0.6)
    ax.tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

#### 4.2 Correlation Heatmap of Numerical Features
Analyzing multicollinearity among numerical variables (`tenure`, `MonthlyCharges`, `TotalCharges`) and the target variable.

In [ ]:
df.groupby("Churn")[num_cols].mean()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df[num_cols + ['Churn']].corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

One-hot encoding

In [ ]:
df_encoded = pd.get_dummies(df, drop_first = True, dtype = int)

In [ ]:
df_encoded.head()

In [ ]:
[col for col in df_encoded.columns if "Churn" in col]

In [ ]:
X = df_encoded.drop(columns="Churn_1")
y = df_encoded["Churn_1"]

### 5. Train-Test Split
We split the dataset into 80% training and 20% testing sets. `stratify=y` is applied to preserve the minority class (churn) proportion across both partitions.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 6. Feature Scaling
We scale numerical features using `StandardScaler`. To prevent data leakage, the scaler is fitted strictly on the training set and then used to transform the test set.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

### 7. Baseline Model: Logistic Regression
Since the dataset is imbalanced (~26.5% churn), we configure the model with `class_weight='balanced'`. This assigns higher penalties to misclassifications of the minority class, ensuring the model does not achieve deceptive accuracy by simply predicting non-churn.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

#### Evaluating our model.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_proba))

## Key Findings & Business Insights
1. **Contract Type is Crucial:** Month-to-month contracts exhibit the highest churn rate (~42.7%), whereas one-year and two-year contracts have significantly lower churn rates (~11% and ~2.8%). The business should incentivize longer-term commitments.
2. **Fiber Optic & Electronic Check Risk:** Customers using Fiber Optic internet and Electronic Check payment methods show above-average churn rates. Customer support and service reliability in these segments should be investigated.
3. **Tenure & Pricing:** Churned customers have a significantly lower median tenure (~18 months) and higher monthly charges compared to retained customers, indicating early lifecycle vulnerability.
4. **Model Performance:** The baseline Logistic Regression model achieves a **ROC-AUC of ~0.84** and a **Recall of ~0.79** for the churn class, successfully identifying the majority of churners while maintaining acceptable discrimination ability.